# NoteWeaver: Read, Review, and Save a Lecture Note

**Member 4 integration demo | English instructions | No AI account required**

Use a real three-page sample lecture to select an original page, preview it,
write and edit a personal note, confirm it, and save and reopen the project.
The lecture contains English and Traditional Chinese to demonstrate bilingual input.
The personal note in this walkthrough is in English.

This notebook uses the implemented reader, note-management, and persistence APIs.
It does **not** generate AI explanations or insert/export supplementary PDF pages.
Those modules will be integrated when available.


## Before you start

Follow [the notebook setup guide](../docs/NOTEBOOK_DEMO.md) and select the kernel
for the environment where you installed this checkout with its preview dependencies.
Keep this notebook inside the cloned repository. It works when the kernel starts
in the repository root or the `examples` directory.

Run cells from top to bottom. **Restart Kernel and Run All Cells** replays the
scripted demonstration, including its sample approval decisions. For your own
notes, read the content before running an approval cell.

Each full run creates a new folder under ignored `generated/notebook-demo/`.
The source lecture is never edited. Notebook outputs are intentionally cleared
in Git; execute the notebook to see the preview and results.


In [ ]:
from html import escape
from pathlib import Path
import tempfile

from IPython.display import HTML, Image, display
from noteweaver import (
    ContractError, add_manual_note, approve_note, create_project,
    document_id_for_file, list_notes, load_project, save_project,
    update_note, validate_page, verify_source,
)
from noteweaver.pdf_reader import PDFReader

candidates = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
repo_root = next(
    (path for path in candidates
     if (path / "src/noteweaver/models.py").is_file()
     and (path / "examples/data/sample_lecture_3p.pdf").is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError("Open this notebook inside the cloned NoteWeaver repository.")

source_pdf = repo_root / "examples/data/sample_lecture_3p.pdf"
output_root = repo_root / "generated/notebook-demo"
output_root.mkdir(parents=True, exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix="run-", dir=output_root))
source_id_before = document_id_for_file(source_pdf)
reader = PDFReader(preview_dir=run_dir / "previews")


def show_note(note):
    # Escape text before displaying it as HTML, including user-authored content.
    body = "\n\n".join(section.content for section in note.sections)
    display(HTML(
        f"<h3>{escape(note.title)}</h3>"
        f"<p>Original page: {note.original_page} | Status: {escape(note.status)}"
        f" | Language: {escape(note.language)}</p>"
        f"<pre style='white-space:pre-wrap'>{escape(body)}</pre>"
    ))


print("Ready: local manual-note workflow; no model requests.")
print("Run folder:", run_dir.relative_to(repo_root))


## 1. Open the lecture and select an original page

`inspect()` reads real PDF metadata and hashes the source bytes. Page numbers are
**1-based original PDF pages**. Here, page 2 covers opportunity cost.


In [ ]:
document = reader.inspect(source_pdf)
selected_page = 2
page = reader.read_page(source_pdf, selected_page)
validate_page(document, page)

assert document.page_count == 3
assert page.original_page == selected_page
assert page.extraction_status == "ok"
assert "Opportunity cost" in page.text
assert "機會成本" in page.text  # Bilingual fixture content, not a translation step.
print("Lecture:", document.filename)
print("Total pages:", document.page_count)
print("Selected original page:", selected_page)
print("Text extraction:", page.extraction_status)
print("\nExtracted lecture text:\n" + page.text)


## 2. Preview the actual PDF page

The image below is rendered from the PDF. If preview rendering is unavailable,
`read_page()` emits a warning and returns `preview_path=None`; extracted text
remains available. Follow the setup guide to install the preview dependencies.
An unavailable preview is never replaced by a fake image.


In [ ]:
if page.preview_path is not None:
    display(Image(filename=page.preview_path, width=800))
else:
    print("Preview unavailable. Read the extracted text above and check the warning.")


## 3. Create a project and write a personal note

This sample note is **written manually**, not generated by AI. Its example is a
personal illustration, not a claim copied from the lecture. New notes are drafts.

Every editing operation returns a new `ProjectState`; assign the return value
back to `project`. The original page anchor connects this note to lecture page 2.


In [ ]:
project = create_project(document, source_pdf)
project = add_manual_note(
    project,
    original_page=selected_page,
    title="Understanding opportunity cost",
    content=(
        "Opportunity cost is the value of the next-best alternative I give up.\n\n"
        "Personal example: if I study instead of taking a paid shift, "
        "the forgone wages are part of the opportunity cost of studying."
    ),
    language="en",
)
note_id = project.notes[-1].note_id
assert project.notes[-1].status == "draft"
assert project.notes[-1].original_page == selected_page
assert project.notes[-1].sections[0].origin == "user"
show_note(project.notes[-1])


## 4. Edit the draft before confirming it

Change `revised_content` to try your own wording. `content=` replaces the entire
note body. The note ID and original-page anchor stay the same.


In [ ]:
revised_content = (
    "Opportunity cost is the value of the best alternative forgone, "
    "not the sum of every alternative.\n\n"
    "Personal example: choosing two hours of revision over a paid shift "
    "means giving up the value of that shift. Time matters even when "
    "no money is spent."
)
project = update_note(project, note_id, content=revised_content)
note = list_notes(project, original_page=selected_page)[0]
assert note.note_id == note_id
assert note.original_page == selected_page
assert note.status == "draft"
show_note(note)


## 5. Confirm the reviewed note

**Review the note above before running this cell.** Running it records your
confirmation for this sample note. Approval records review state, not a guarantee
of factual correctness. The separate PDF exporter consumes approved notes; this cell
does not export or modify a PDF.


In [ ]:
project = approve_note(project, note_id)
approved = list_notes(project, status="approved")
assert len(approved) == 1
assert approved[0].note_id == note_id
show_note(approved[0])
print("Approved notes:", len(approved))


## 6. Save and reopen the project

The JSON stores editable note data and the source reference, not the PDF or its
preview image. Reopening verifies that the source still has the same fingerprint.
The source PDF must remain available.

The first save refuses to overwrite an existing file. If you rerun just this
cell after saving, `FileExistsError` is expected; restart and run all for a new
run folder, or use the explicit update-save example below for this demo file.


In [ ]:
project_file = save_project(project, run_dir / "study.json")
reopened = load_project(project_file)
verify_source(reopened, source_pdf)

assert reopened == project
assert reopened.notes[0].note_id == note_id
assert reopened.notes[0].original_page == selected_page
assert reopened.notes[0].status == "approved"
assert reopened.notes[0].sections[0].content == revised_content
print("Saved:", project_file.relative_to(repo_root))
print("Reload verified: content, note ID, page anchor, language, and approval preserved.")
show_note(reopened.notes[0])


## 7. A later edit requires another review

Editing an approved note resets it to **draft**. Saving and reopening must preserve
that draft state, so an old approval cannot silently apply to new wording.
`overwrite=True` below intentionally updates only this run's demo JSON.


In [ ]:
project = update_note(reopened, note_id, title="Opportunity cost: my revision note")
assert project.notes[0].status == "draft"
assert len(list_notes(project, status="approved")) == 0
save_project(project, project_file, overwrite=True)
project = load_project(project_file)
assert project.notes[0].status == "draft"
show_note(project.notes[0])
print("After editing and reopening: no approved notes; another review is required.")


Review the revised title and content above. Run the next cell to confirm this
sample note again and persist the new approval.


In [ ]:
project = approve_note(project, note_id)
save_project(project, project_file, overwrite=True)
final_project = load_project(project_file)
assert final_project == project
assert final_project.notes[0].status == "approved"
print("Reviewed again, saved, and reopened successfully.")


## 8. Verify a clear error and preserve existing work

A note cannot target a page outside the lecture. The following intentional error
is caught and displayed. The existing project must remain unchanged.


In [ ]:
before_invalid_operation = final_project
try:
    add_manual_note(
        final_project, original_page=document.page_count + 1,
        title="Invalid page example", content="This note must be rejected.",
        language="en",
    )
except ContractError as error:
    print("Expected validation error:", error)
else:
    raise AssertionError("An out-of-range original page was accepted.")

assert final_project == before_invalid_operation
assert load_project(project_file) == final_project
assert document_id_for_file(source_pdf) == source_id_before
verify_source(final_project, source_pdf)
print("Existing project unchanged. Original lecture bytes unchanged.")
print("Demo complete: one approved English note remains attached to original page 2.")
print("Saved project:", project_file.relative_to(repo_root))


## What this demonstrates and what comes next

- **Member 1:** real PDF metadata, bilingual text extraction, and original-page preview.
- **Member 4:** project creation, manual notes, editing, approval, filtering,
  source-verified save/reload, and error-safe integration.
- **Member 2 handoff:** connect explanation generation to the existing shared note format.
- **Member 3 handoff:** render approved notes and insert them after their original pages.

The saved output is a **project JSON**, not an exported study PDF. No AI key is
needed and no model call is made. Use the separate export API or examples/export_demo.py for PDF output.

Run folders stay under `generated/notebook-demo/` for inspection. You may delete
a run folder when finished with its demo JSON and preview. Clear notebook outputs
before committing, especially after experimenting with personal lecture material.
